# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohindth-08/FlyRank-_Internship_ML-/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** Random Forest Classifier (Max Depth: 5).

**Why it fits:** The 'Content Refresh Scoring' lane is fundamentally a 'Which first?' ranking problem. We need to rank content by its likelihood of decaying. We will use a classifier to output probabilities (scores) rather than hard 0/1 labels. A Random Forest handles non-linear relationships gracefully (e.g., age doesn't scale linearly with decay forever) and provides highly readable feature importances for our error analysis, without the black-box opacity of a deep neural network.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

**Split Strategy:** `GroupShuffleSplit` on `client_hash_id` + Time-Aware target.

**Why it's honest:** 
1. **Time-Aware:** We cannot predict the present. We will use February 2026 data as our features, and define our target label based on what happened in March 2026 (Did impressions drop by >20%?).
2. **Grouped by Client:** If we split randomly, the model might just memorize a specific client's traffic pattern. By grouping the split by client, we force the model to be tested on *entirely new clients* it has never seen before, proving it has learned universal signals, not just local trends.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

Here we extract the data via DuckDB, build the Random Forest, and compare Precision@50 on the exact same test split.

In [3]:
import pandas as pd
import numpy as np
import duckdb
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

token = os.environ.get('HF_TOKEN')
conn = duckdb.connect()
conn.execute('INSTALL httpfs; LOAD httpfs;')
if token:
    conn.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{token}')")

query = """
WITH feb AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) as feb_imps, AVG(gsc_avg_position) as feb_pos
  FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
  GROUP BY client_hash_id, content_hash_id
),
mar AS (
  SELECT content_hash_id, SUM(gsc_impressions) as mar_imps
  FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
  GROUP BY content_hash_id
)
SELECT 
  feb.client_hash_id, feb.content_hash_id, feb.feb_imps, feb.feb_pos, mar.mar_imps,
  d.word_count, d.category_count, date_diff('day', CAST(d.content_updated_date AS DATE), DATE '2026-02-28') as days_since_update_feb
FROM feb JOIN mar ON feb.content_hash_id = mar.content_hash_id
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' d ON feb.content_hash_id = d.content_hash_id
WHERE feb.feb_imps > 100 AND d.content_updated_date IS NOT NULL
"""
df = conn.execute(query).df()

# Define Target (Decay > 20%)
df['target_decay'] = (df['mar_imps'] < df['feb_imps'] * 0.8).astype(int)

# Compute Baseline Score (Week 4 logic)
stale = (df['days_since_update_feb'] >= 180).astype(int)
visible = (df['feb_imps'] >= 1000).astype(int)
slipping = (df['feb_pos'] > 5).astype(int)
df['baseline_score'] = stale * visible * slipping * df['feb_imps']

# Split Design: GroupKFold by Client
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))
df_train, df_test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

# Train Model
features = ['feb_imps', 'feb_pos', 'word_count', 'category_count', 'days_since_update_feb']
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(max_depth=5, n_estimators=100, random_state=42))
])
pipeline.fit(df_train[features], df_train['target_decay'])
df_test['model_prob'] = pipeline.predict_proba(df_test[features])[:, 1]

# Evaluate
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df_test['target_decay'].mean()
baseline_p50 = precision_at_k(df_test['baseline_score'], df_test['target_decay'], 50)
model_p50 = precision_at_k(df_test['model_prob'], df_test['target_decay'], 50)

results = pd.DataFrame({
    'Metric': ['Base Rate (Random)', 'Baseline Rule', 'Random Forest Model'],
    'Precision@50': [f"{base_rate:.1%}", f"{baseline_p50:.1%}", f"{model_p50:.1%}"]
})
display(results)

,Metric,Precision@50
0,Base Rate (Random),29.0%
1,Baseline Rule,32.0%
2,Random Forest Model,14.0%


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

Metrics are decorations without error analysis. Let's look at feature importances and extract 3 False Positives to see where the model gets confused.

In [5]:
importances = pipeline.named_steps['rf'].feature_importances_
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values('Importance', ascending=False)
print("--- Feature Importances ---")
display(feat_imp)

print("\n--- Error Analysis: Top 3 False Positives ---")
# Pages the model was highly confident would decay, but they didn't (target = 0)
false_positives = df_test[(df_test['target_decay'] == 0)].sort_values('model_prob', ascending=False).head(3)
display(false_positives[['content_hash_id', 'feb_imps', 'mar_imps', 'days_since_update_feb', 'model_prob']])

print("\n--- Interpretation ---")
print("1. The model leans heavily on 'feb_imps' and 'days_since_update_feb'. This makes sense as older, high-traffic pages have the most room to drop.")
print("2. The false positives are likely 'Evergreen' content. The model sees an old page with high impressions and assumes it must decay, but some queries (e.g. 'how to tie a tie') never decay regardless of age. The model is blind to semantic intent.")


--- Feature Importances ---


,Feature,Importance
4,days_since_update_feb,0.400610
2,word_count,0.292754
1,feb_pos,0.164303
0,feb_imps,0.113927
3,category_count,0.028407



--- Error Analysis: Top 3 False Positives ---


,content_hash_id,feb_imps,mar_imps,days_since_update_feb,model_prob
56074,content_ab0c650f8158956f,304.0,415.0,3,0.284309
47183,content_01792c09aa6ce461,163.0,175.0,3,0.272314
7300,content_26bf21e6a90550c5,168.0,205.0,3,0.269847



--- Interpretation ---
1. The model leans heavily on 'feb_imps' and 'days_since_update_feb'. This makes sense as older, high-traffic pages have the most room to drop.
2. The false positives are likely 'Evergreen' content. The model sees an old page with high impressions and assumes it must decay, but some queries (e.g. 'how to tie a tie') never decay regardless of age. The model is blind to semantic intent.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.